# 📗 질문에 맞는 검색 도구를 LangGraph로 연결합니다

교안 01에서 State·노드·엣지를 배우고, 교안 02에서 Jev 분류와 하이브리드 검색을 연결했습니다. 이번 교안은 이를 직접 작성하는 **통합 실습**입니다. 질문에 따라 하이브리드 검색, 관계 조회 또는 두 검색을 함께 실행합니다.

**완성할 결과물:** Jev가 검색 경로를 정하고, 필요한 도구로 근거를 모아 인용 답변을 돌려주는 `routing_graph`입니다.


<img src="images/19_lesson03_compiled_graph.png" width="1100" style="background:white;display:block;margin:16px auto" alt="① 질문 분류부터 ⑥ 근거 답변까지의 개요. 분기 A는 검색 종류를, 분기 B는 추가 관계 조회 여부를 고릅니다.">



**①~⑥은 노드, A·B는 조건부 분기 위치**입니다. 실선은 고정 연결, 점선은 조건부 연결입니다. 각 절에서 같은 번호의 부분을 작성합니다.

## 이번 교안의 목표

1. **State 생성:** 질문·경로·검색 결과·답변을 전달할 필드를 정합니다.
2. **노드 생성:** 분류·질문 분리·검색·근거 수집·답변을 각각 함수로 만듭니다.
3. **엣지 연결:** 두 라우팅 함수와 조건부 엣지로 필요한 검색을 연결합니다.
4. **실행·확인:** 실제 질문의 경로·Cypher·답변을 원문과 대조합니다.

**작성 방식:** 긴 프롬프트·출력 스키마·검색 도구는 제공합니다. `변수명 = ...`의 우변과 연결문을 채우세요. 각 노드를 준비한 뒤 마지막에 그래프를 조립합니다.

## 준비 A. 라이브러리와 영화 데이터를 적재합니다

지난 단원 교안 03의 모델·DB 연결과 영화·인물·원문·청크·배포 벡터 적재를 사용합니다. 저장소 루트에서 `uv sync --frozen`을 실행하고 `.env`의 Neo4j·GPT·Jev 설정을 준비하세요. 모델은 `OPENAI_CHAT_MODEL=gpt-5.6-luna`, `TYPESAFE_MODEL=jev-1.13.0`입니다.


<img src="images/15_lesson03_loaded_graph.png" width="1100" style="background:white;display:block;margin:16px auto" alt="청크·원문·영화·인물의 연결과 전문·벡터 인덱스 대상">



영화 청크에는 `Chunk`와 `Day49MovieChunk` 레이블을 함께 붙이고, 검색 인덱스는 영화 전용 `Day49MovieChunk`를 대상으로 만듭니다. 문서 벡터는 `data/embeddings/movies_complete.npz`에서 읽으므로 문서 임베딩 API를 호출하지 않습니다.

기존 적재 방식대로 같은 대상의 검색 인덱스와 공용 `Chunk.embedding` 벡터 인덱스를 정리하고 영화 전용 인덱스를 만듭니다. 노드·관계·저장된 벡터는 유지합니다. 이전 실습의 공용 인덱스를 다시 쓰려면 해당 인덱스 생성 셀을 실행하세요.


In [ ]:
# 모델과 DB 연결에 필요한 라이브러리를 불러옵니다.
import json
import os
import re
import sys
from pathlib import Path
from IPython.display import Markdown, display
from urllib.parse import urlparse

import pandas as pd
from dotenv import find_dotenv, load_dotenv
from neo4j import GraphDatabase
from langchain_openai import ChatOpenAI, OpenAIEmbeddings



# 노트북 기준 경로입니다. 정답은 상위 폴더를 사용합니다.
material_dir = Path(".")
data_dir = material_dir / "data"
sys.path.insert(0, str(material_dir.resolve()))
from graph_data import load_graph, read_vectors


# 공통 설정을 읽고 교안의 .env를 우선 적용합니다.
load_dotenv(find_dotenv(usecwd=True))
load_dotenv(material_dir.resolve().parent / ".env")
load_dotenv(material_dir / ".env", override=True)
# 모델만 준비합니다. API 호출은 검색하거나 답변할 때 합니다.
embedding_model = OpenAIEmbeddings(
    model="text-embedding-3-large", dimensions=768,
    check_embedding_ctx_length=False,  # 준비한 청크를 자동으로 나누지 않습니다.
)
llm = ChatOpenAI(model=os.getenv("OPENAI_CHAT_MODEL", "gpt-5.6-luna"))

# .env의 주소와 계정으로 Neo4j에 연결합니다.
neo4j_uri = os.environ["NEO4J_URI"]
driver = GraphDatabase.driver(
    neo4j_uri, auth=(os.environ["NEO4J_USER"], os.environ["NEO4J_PASSWORD"])
)
driver.verify_connectivity()
address = urlparse(neo4j_uri)
print("Neo4j 연결:", address.hostname, address.port or 7687)


def run_cypher(query, **params):
    """Cypher 결과를 행 딕셔너리의 리스트로 반환합니다."""
    # 인자를 Cypher의 $파라미터로 전달합니다.
    with driver.session() as session:
        # 조회 결과를 딕셔너리 목록으로 반환합니다.
        return [record.data() for record in session.run(query, **params)]


# 영화 그래프와 원문, 기존 청크 벡터를 불러옵니다.
movies = load_graph(data_dir / "movies_extraction_packet.json")
chunk_ids, vectors = read_vectors(
    data_dir / "embeddings/movies_complete.npz", movies["chunks"], embedding_model
)
# 검색 결과의 원문 ID로 줄거리를 찾을 딕셔너리입니다.
documents_by_id = movies["documents"]
print("개체:", len(movies["nodes"]), "관계:", len(movies["relations"]))
print("문서:", len(documents_by_id), "청크:", len(chunk_ids), "벡터:", vectors.shape)


# 이전 실습의 공용 Chunk.embedding 벡터 인덱스만 정리합니다.
# 노드, 관계, 저장된 임베딩은 유지됩니다.
indexes = run_cypher("""
    SHOW VECTOR INDEXES
    YIELD name, labelsOrTypes, properties
    WHERE labelsOrTypes = ['Chunk'] AND properties = ['embedding']
    RETURN name
""")

for index in indexes:
    name = index["name"].replace("`", "``")
    run_cypher(f"DROP INDEX `{name}` IF EXISTS")
    print("삭제:", index["name"])

# 개체의 standard_id 중복을 막는 고유 제약을 만듭니다.
run_cypher("CREATE CONSTRAINT rag_entity_id IF NOT EXISTS FOR (n:RAGEntity) REQUIRE n.standard_id IS UNIQUE")
# 원문 id 중복을 막는 고유 제약을 만듭니다.
run_cypher("CREATE CONSTRAINT document_id IF NOT EXISTS FOR (n:Document) REQUIRE n.id IS UNIQUE")
# 청크 id 중복을 막는 고유 제약을 만듭니다.
run_cypher("CREATE CONSTRAINT chunk_id IF NOT EXISTS FOR (n:Chunk) REQUIRE n.id IS UNIQUE")

# 영화와 인물을 종류별로 적재합니다.
for entity_type in ["Movie", "Person"]:
    # 현재 종류의 개체만 고릅니다.
    rows = [row for row in movies["nodes"] if row["entity_type"] == entity_type]
    # 현재 종류의 영화 또는 인물 노드를 만들고 속성을 갱신합니다.
    run_cypher(f"""
        // 목록을 개체별 행으로 펼칩니다.
        UNWIND $rows AS item
        // 같은 ID의 노드를 재사용하고, 없으면 생성합니다.
        MERGE (n:RAGEntity {{standard_id: item.standard_id}})
        // 종류 레이블을 추가하고 속성·이름을 갱신합니다.
        SET n:{entity_type}, n += item.properties, n.name = item.name
    """, rows=rows)

# 적재한 노드 사이에 종류별 관계를 연결합니다.
for relation in ["ACTED_IN", "DIRECTED", "PRODUCED", "WROTE", "REVIEWED", "FOLLOWS"]:
    # 현재 유형의 관계만 고릅니다.
    rows = [row for row in movies["relations"] if row["relation"] == relation]
    # 현재 유형의 관계를 연결하고 근거와 출처를 저장합니다.
    run_cypher(f"""
        // 목록을 관계별 행으로 펼칩니다.
        UNWIND $rows AS item
        // 관계의 시작·끝 노드를 ID로 찾습니다.
        MATCH (a:RAGEntity {{standard_id: item.subject_id}})
        MATCH (b:RAGEntity {{standard_id: item.object_id}})
        // 같은 유형·방향·근거 ID의 관계가 없으면 만들고, 있으면 재사용합니다.
        MERGE (a)-[r:{relation} {{claim_id: item.claim_id}}]->(b)
        // 관계 속성과 근거·출처를 저장합니다.
        SET r += item.properties, r.evidence = item.evidence,
            r.source_url = item.source_url, r.source_doc_id = item.source_doc_id
    """, rows=rows)


# 원문 노드를 먼저 저장합니다.
run_cypher("""
    // 원문 목록을 한 문서당 한 행으로 펼칩니다.
    UNWIND $rows AS item
    // 같은 ID의 원문 노드가 없으면 만들고, 있으면 재사용합니다.
    MERGE (d:Document {id: item.doc_id})
    // 제목·본문·출처 URL을 갱신합니다.
    SET d.title = item.title, d.text = item.text, d.url = item.url
""", rows=list(documents_by_id.values()))

# 원문과 해당 영화를 연결합니다.
run_cypher("""
    // doc_id로 원문을, standard_id로 해당 영화를 찾습니다.
    UNWIND $rows AS item
    MATCH (d:Document {id: item.doc_id})
    MATCH (m:Movie {standard_id: item.standard_id})
    // 원문과 해당 영화를 ABOUT_MOVIE 관계로 연결합니다.
    MERGE (d)-[:ABOUT_MOVIE]->(m)
""", rows=movies["document_links"])

# 청크와 벡터의 순서를 맞춰 전체 저장 목록을 만듭니다.
rows = []
for index, chunk in enumerate(movies["chunks"]):
    # NumPy 벡터를 DB에 전달할 목록으로 바꿉니다.
    rows.append({"id": chunk_ids[index], "doc_id": chunk.metadata["source_doc_id"],
                 "text": chunk.page_content, "embedding": vectors[index].tolist()})

# 전체 청크와 벡터를 한 번에 저장합니다.
run_cypher("""
    UNWIND $rows AS item
    // 원문을 찾고 청크를 ID로 생성·재사용합니다.
    MATCH (d:Document {id: item.doc_id})
    MERGE (c:Chunk {id: item.id})
    // 이 교안의 영화 청크만 검색하도록 전용 레이블을 붙입니다.
    SET c:Day49MovieChunk
    // text는 전문 검색, embedding은 벡터 검색, source_id는 원문별 결과 통합에 씁니다.
    SET c.text = item.text, c.source_id = item.doc_id, c.embedding = item.embedding
    // 청크와 원문을 FROM_DOCUMENT 관계로 연결합니다.
    MERGE (c)-[:FROM_DOCUMENT]->(d)
""", rows=rows)


# 같은 이름의 옛 설정과 같은 검색 대상의 다른 이름 인덱스를 정리합니다.
# IF NOT EXISTS만 쓰면 다른 이름의 인덱스 때문에 필요한 이름이 생성되지 않을 수 있습니다.
indexes = run_cypher("""
    SHOW INDEXES YIELD name, type, entityType, labelsOrTypes, properties
    WHERE name IN ['movie_chunks', 'movie_fulltext']
       OR (entityType = 'NODE' AND labelsOrTypes = ['Day49MovieChunk']
           AND ((type = 'VECTOR' AND properties = ['embedding'])
             OR (type = 'FULLTEXT' AND properties = ['text'])))
    RETURN name
""")
for index in indexes:
    name = index["name"].replace("`", "``")
    run_cypher(f"DROP INDEX `{name}` IF EXISTS")
# 아래에서 정해진 이름, 차원, 분석기로 다시 만듭니다. 노드와 저장된 벡터는 유지됩니다.

# 벡터 유사도 검색용 인덱스를 만듭니다.
run_cypher("""
    CREATE VECTOR INDEX movie_chunks IF NOT EXISTS
    // 이 교안의 영화 청크만 벡터 검색 대상으로 지정합니다.
    FOR (c:Day49MovieChunk) ON c.embedding
    // 저장된 벡터의 768차원과 코사인 유사도를 설정합니다.
    OPTIONS {indexConfig: {`vector.dimensions`: 768, `vector.similarity_function`: 'cosine'}}
""")
# 벡터 인덱스가 검색 가능한 상태가 될 때까지 최대 300초 기다립니다.
run_cypher("CALL db.awaitIndex('movie_chunks', 300)")
# 단어 일치 검색용 전문 인덱스를 만듭니다.
run_cypher("""
    CREATE FULLTEXT INDEX movie_fulltext IF NOT EXISTS
    // 같은 영화 청크의 text만 전문 검색 대상으로 지정합니다.
    FOR (c:Day49MovieChunk) ON EACH [c.text]
    // 영어 분석기를 사용하고, 본문 변경을 커밋 시 인덱스에 반영합니다.
    OPTIONS {indexConfig: {`fulltext.analyzer`: 'english', `fulltext.eventually_consistent`: false}}
""")
# 전문 인덱스가 검색 가능한 상태가 될 때까지 최대 300초 기다립니다.
run_cypher("CALL db.awaitIndex('movie_fulltext', 300)")
print("기존 문서 임베딩을 적재했습니다. 문서 임베딩 API 호출은 없습니다.")


## 준비 B. 공통 검색 도구를 준비합니다

교안 02의 하이브리드 검색을 Neo4j의 전문·벡터 검색으로 확장합니다. 아래 셀은 검색 도구와 함수를 준비하며, 질문 검색은 뒤에서 그래프를 실행할 때 시작합니다.


<img src="images/16_lesson03_jev_language.png" width="1100" style="background:white;display:block;margin:16px auto" alt="Jev가 질문 언어를 판단하고 한글 질문만 영어 핵심어로 변환해 전문 검색하는 흐름">



- `fulltext_documents`: Jev가 한글로 판단한 질문을 영어 검색어로 바꿔 전문 검색합니다.
- `ensemble_retriever`: 전문·벡터 검색 순위를 RRF로 결합합니다. 기존 가중치 0.4·0.6을 사용합니다.
- `collect_evidence`: 원문·조회 사실·관계 근거를 모읍니다.

그림의 **언어 판단**과 뒤의 **검색 경로 판단**은 서로 다른 `Choice` 기준을 사용합니다.


In [ ]:
from langchain_typesafe import Choice, TypeSafeClassifier
from neo4j_graphrag.retrievers import VectorRetriever
from neo4j_graphrag.types import RetrieverResultItem
from langchain_core.documents import Document
from langchain_core.runnables import RunnableLambda
from langchain_classic.retrievers import EnsembleRetriever
from pydantic import BaseModel, Field
from langchain_core.prompts import ChatPromptTemplate

def format_vector(record):
    """검색된 청크의 본문과 원문 ID·유사도 점수를 반환합니다."""
    node = record["node"]
    # content는 청크 본문, metadata는 청크 ID, 원문 ID, 유사도를 담는 딕셔너리입니다.
    return RetrieverResultItem(content=node["text"], metadata={
        "chunk_id": node["id"], "source_id": node["source_id"], "score": record["score"]
    })


# 벡터 인덱스와 결과 변환 함수를 연결해 벡터 검색기를 만듭니다.
vector_retriever = VectorRetriever(
    driver, index_name="movie_chunks", embedder=embedding_model,
    result_formatter=format_vector,
)


def vector_documents(query_vector, top_k=12):
    """검색된 청크 중 영화별 최고 점수 한 건을 대표 후보로 반환합니다."""
    # 질문 벡터와 유사한 청크를 점수 내림차순으로 검색합니다.
    result = vector_retriever.search(query_vector=query_vector, top_k=top_k)
    # 영화별 후보 목록을 만들기 위해 처음 나온 최고 점수 청크를 대표로 남깁니다.
    rows, seen = [], set()
    for item in result.items:
        row = item.metadata
        if row["source_id"] not in seen:
            seen.add(row["source_id"])
            rows.append(dict(row))
    return rows


# Jev가 한글과 영어 중 질문의 언어를 선택하도록 설정합니다.
language_classifier = TypeSafeClassifier(model=os.getenv("TYPESAFE_MODEL", "jev-1.13.0"))
language_question = Choice(
    instructions="질문의 주된 언어를 선택하세요.",
    criteria={"ko": "한글 질문. 영어 영화명 등이 섞인 한글 문장도 포함합니다.",
              "en": "한글 없이 영어 단어나 문장으로 작성된 질문입니다."},
)


# 영어 검색어를 문자열 목록으로 받습니다.
class EnglishKeywords(BaseModel):
    """영어 원문 검색에 사용할 핵심어 목록입니다."""
    keywords: list[str] = Field(description="중복 없는 영어 단어 4~8개")


# 한글 질문에서 영어 검색어 목록을 만드는 체인입니다.
lexical_prompt = ChatPromptTemplate.from_messages([
    ("system", "한글 영화 줄거리 질문을 영어 원문 검색용 핵심어 4~8개로 바꾸세요. "
               "질문의 핵심 사건·대상을 나타내는 단어를 골라 keywords의 각 항목에 영어 단어 하나씩 담으세요. "
               "영화 정답이나 배우를 추측해 추가하지 마세요. 질문에 없는 고유명사는 넣지 마세요."),
    ("human", "{question}"),
])
lexical_chain = lexical_prompt | llm.with_structured_output(EnglishKeywords)


def fulltext_documents(query, top_k=12):
    """질문의 언어에 맞춰 전문 검색하고 원문별 최고 점수 청크를 반환합니다."""
    # state에 질문, questions에 언어 선택 기준을 전달합니다.
    decision = language_classifier.invoke({
        "state": query, "questions": {"language": language_question},
    })
    language = decision.choices["language"].choice
    # 한글은 영어 검색어 목록을 받아 OR로 연결하고, 영어는 그대로 사용합니다.
    if language == "ko":
        terms = lexical_chain.invoke({"question": query})
        search_text = " OR ".join(f'"{word}"' for word in terms.keywords)
    else:
        search_text = query
    print("질문 언어:", language, "| 전문 검색어:", search_text)
    # 전문 인덱스에서 검색어와 일치하는 청크를 최대 top_k개 조회합니다.
    hits = run_cypher("""
        // 전문 인덱스에서 관련 청크를 최대 top_k개 조회합니다.
        CALL db.index.fulltext.queryNodes('movie_fulltext', $search_text, {limit: $top_k})
        YIELD node, score
        // 결과 통합에 쓸 원문 ID를 반환합니다.
        RETURN node.source_id AS source_id, node.id AS chunk_id, score
        // 점수 내림차순으로 정렬합니다.
        ORDER BY score DESC, chunk_id
    """, search_text=search_text, top_k=top_k)
    rows, seen = [], set()
    # 영화별 순위를 비교하도록 원문마다 최고 점수 검색 행을 대표로 남깁니다.
    for row in hits:
        source_id = row["source_id"]
        if source_id not in seen:
            seen.add(source_id)
            rows.append(row)
    return rows




def as_documents(rows):
    """원문별 검색 결과를 LangChain Document 목록으로 바꿉니다."""
    return [Document(page_content=documents_by_id[row["source_id"]]["text"],
                     metadata={"source_id": row["source_id"]}) for row in rows]


# 기존 검색 함수를 LangChain에서 호출할 수 있도록 연결합니다.
# 두 함수는 이미 원문별 중복을 제거한 결과를 반환합니다.
fulltext_retriever = RunnableLambda(
    lambda query: as_documents(fulltext_documents(query, top_k=30))
)
dense_retriever = RunnableLambda(
    lambda query: as_documents(vector_documents(embedding_model.embed_query(query), top_k=30))
)
# 전문 검색과 벡터 검색을 가중 RRF로 결합합니다. 같은 원문은 source_id로 합칩니다.
ensemble_retriever = EnsembleRetriever(
    retrievers=[fulltext_retriever, dense_retriever],
    weights=[0.4, 0.6], c=60, id_key="source_id",
)

# 실제 저장된 관계 ID로 근거 문장을 찾습니다.
relations_by_id = {row["claim_id"]: row for row in movies["relations"]}


def collect_evidence(graph_rows, source_ids=()):
    """선택한 원문·그래프 조회 사실·관계 근거를 하나의 목록으로 모읍니다."""
    evidence = []
    for source_id in source_ids:
        doc = documents_by_id[source_id]
        evidence.append({"evidence_id": source_id, "kind": "plot", "title": doc["title"],
                         "url": doc["url"], "text": doc["text"]})
    claim_ids = set()
    for i, row in enumerate(graph_rows, start=1):
        evidence.append({"evidence_id": f"graph:{i}", "kind": "graph_record",
                         "text": json.dumps(row["facts"], ensure_ascii=False)})
        claim_ids.update(row["claim_ids"])
    for claim_id in sorted(claim_ids):
        evidence.append({"evidence_id": claim_id, "kind": "relation",
                         "text": relations_by_id[claim_id]["evidence"]})
    return evidence


## 1. State를 생성합니다

**State는 노드 사이에서 공유하는 데이터**입니다. 질문을 받고, 선택한 경로와 각 검색 결과를 담아 마지막 답변까지 전달합니다. 모델·DB·검색기 객체는 State 밖에서 준비합니다.

| 필드 | 담을 정보 |
|---|---|
| question, route | 원 질문과 선택한 검색 경로 |
| semantic_query, graph_request | 줄거리 검색 입력과 관계 조회 입력 |
| documents | 하이브리드 검색 문서 |
| graph_rows, cypher | 관계 조회 행과 생성 쿼리 |
| source_ids, evidence | 선택한 원문 ID와 답변 근거 |
| answer, evidence_ids | 답변과 인용 ID |

노드는 읽은 State 전체가 아니라 **갱신할 키와 새 값**을 반환합니다. 반환하지 않은 필드는 유지됩니다. 검색 결과를 누적하지 않으므로 각 필드는 기본 규칙대로 새 값으로 교체합니다.


In [ ]:
from typing import TypedDict

# 검색 워크플로의 상태 스키마 정의
class RoutingState(TypedDict):
    question: str  # 검색 경로 분류와 최종 답변에 사용할 원 질문
    route: str  # 필요한 검색을 연결할 경로: hybrid, graph, combined
    semantic_query: str  # 전문 검색과 벡터 검색에 전달할 줄거리 검색어
    graph_request: str  # Cypher 생성에 전달할 관계와 속성 조건
    documents: list[Document]  # 답변에 쓸 원문 후보를 고르기 위한 하이브리드 검색 결과
    graph_rows: list[dict]  # 근거 수집에 쓸 조회 결과: facts(사실), claim_ids(관계 ID 목록)
    cypher: str  # 질문의 조건이 반영됐는지 확인할 생성 쿼리
    source_ids: list[str]  # 원문 근거를 불러올 때 사용할 선택된 문서 ID
    evidence: list[dict]  # GPT가 읽을 근거: evidence_id(인용 ID), kind(근거 유형), text(내용)
    answer: str  # 사용자에게 보여 줄 근거 기반 최종 답변
    evidence_ids: list[str]  # 답변의 인용을 실제 근거와 대조할 때 사용할 ID


### 초기 상태를 준비합니다

입력 구조는 제공합니다. 새 질문마다 빈 `initial_state`에서 시작하고 `question`만 바꿉니다. 이전 실행 결과를 그대로 입력하면 앞 검색의 근거가 남을 수 있습니다.


In [ ]:
# 초기 상태 선언: 키는 State 필드 이름, 값은 해당 필드의 시작 데이터
initial_state: RoutingState = {
    "question": "", "route": "",
    "semantic_query": "", "graph_request": "", "documents": [],
    "graph_rows": [], "cypher": "", "source_ids": [], "evidence": [],
    "answer": "", "evidence_ids": [],
}
new_question = (
    "Sleepless in Seattle과 출연 배우를 최소 2명 공유하고 "
    "1993년 이후(1993년 제외) 개봉한 다른 영화 중 "
    "서로 경쟁하는 서점 주인들이 익명 이메일로 사랑하게 되는 작품을 찾아 "
    "공통 배우와 근거를 보여 주세요."
)


## 2. 노드를 생성합니다

각 노드는 **필요한 State를 읽고 변경한 필드만 반환**합니다. 노드별로 사용할 프롬프트·체인과 함수를 같은 셀에서 준비합니다. 함수 정의만으로 검색이 실행되지는 않습니다.

### 2-1. ① classify: 질문에 필요한 검색을 고릅니다

**읽는 State → 갱신할 State:** question → question, route, semantic_query, graph_request.


<img src="images/13_lesson03_tool_choice.png" width="1100" style="background:white;display:block;margin:16px auto" alt="같은 영화 제목이어도 줄거리 원문 또는 관계·속성 중 필요한 근거에 따라 검색 도구를 고릅니다">



| 경로 | 필요한 근거 | 실행할 작업 |
|---|---|---|
| hybrid | 사건·상황·줄거리 원문 | 전문·벡터 검색 |
| graph | 출연·공유 배우·개봉연도 | 관계·속성 조회 |
| combined | 줄거리와 관계·속성 조건 모두 | 질문 분리 후 두 검색 |

제목이 언급됐다는 이유만으로 graph를 고르지 않습니다. Jev는 필요한 근거를 판단하고, 실제 실행 순서는 3절의 조건부 엣지가 정합니다.

**작성할 부분:**

1. `route_classifier`: `TypeSafeClassifier(model=os.getenv("TYPESAFE_MODEL", "jev-1.13.0"))`로 만듭니다.
2. `decision`: 분류기의 `invoke`에 `{"state": question, "questions": {"route": route_question}}`을 전달합니다.
3. `update`: question, route, semantic_query, graph_request를 담습니다. route는 `decision.choices["route"].choice`, 두 검색 입력은 원 question으로 초기화합니다.


In [ ]:
# 질문 유형 판단도 기존 언어 판단과 같은 Jev 모델을 사용합니다.
route_classifier = ...
route_question = Choice(
    instructions="영화 질문에 필요한 검색 경로를 선택하세요. 영화 제목 언급만으로 graph를 선택하지 마세요.",
    criteria={
        "hybrid": "배우 관계나 개봉연도 조건 없이 줄거리·사건·분위기로 영화를 찾거나 원문 내용을 묻는 질문",
        "graph": "줄거리 조건 없이 배우의 출연·공유 배우·개봉연도 등 그래프 관계나 속성만 조회하는 질문",
        "combined": "줄거리·사건·분위기 조건과 배우 관계·개봉연도 등 그래프 조건을 함께 만족해야 하는 질문",
    },
)


def classify(state: RoutingState):
    """Jev가 고른 경로와 두 검색의 기본 입력을 준비합니다."""
    question = state["question"].strip()
    decision = ...
    # 두 검색 입력을 원 질문으로 초기화: combined 경로에서는 split_query가 갱신
    update = ...
    return update


### 2-2. ② split_query: 복합 질문을 검색별 입력으로 나눕니다

**읽는 State → 갱신할 State:** question → semantic_query, graph_request.

긴 프롬프트·출력 스키마·체인은 제공합니다. 줄거리는 semantic_query, 관계·속성 조건은 graph_request로 나눕니다. 숫자·범위·제외 조건을 유지해야 합니다.

**작성할 부분:** `queries`에는 query_split_chain에 `{"question": state["question"]}`을 전달한 결과를 담습니다. `update`에는 queries의 두 필드를 같은 이름의 State 키로 반환합니다. 이 노드는 combined에서만 실행하도록 연결합니다.


In [ ]:
# 검색별 입력을 정해진 두 필드로 받습니다.
class SearchQueries(BaseModel):
    """원 질문에서 분리한 줄거리 검색어와 그래프 조건 질문입니다."""
    semantic_query: str = Field(description="전문·벡터 검색에 사용할 줄거리의 사건·상황")
    graph_request: str = Field(description="질문에 명시된 모든 그래프 관계·속성 조건을 담은 자연어 질문")


# 줄거리는 내용 검색에, 관계와 속성 조건은 그래프 검색에 전달합니다.
query_split_prompt = ChatPromptTemplate.from_messages([
    ("system", "영화 질문을 두 검색 입력으로 나누세요. "
               "semantic_query에는 줄거리의 사건·상황만 담고 그래프 관계·속성 조건은 빼세요. "
               "graph_request에는 질문에 명시된 모든 그래프 관계·속성 조건과 조회 대상을 보존하세요. 예를 들어 출연·감독·제작 관계, 개봉연도, 인원 수, 제외 대상입니다. "
               "graph_request에는 줄거리 조건을 넣지 말고 Cypher가 아닌 자연어 질문을 작성하세요. "
               "숫자와 이상·초과·이전·이후 등 범위 표현을 바꾸지 마세요. "
               "원 질문에 없는 조건·영화명·인물명을 추가하지 마세요."),
    ("human", "{question}"),
])
query_split_chain = query_split_prompt | llm.with_structured_output(SearchQueries)


def split_query(state: RoutingState):
    """복합 질문의 줄거리와 그래프 조건을 분리합니다."""
    # 두 검색기가 각각 사용할 입력으로 분리
    queries = ...
    update = ...
    return update


### 2-3. ③ search_hybrid: 전문·벡터 검색으로 문서를 찾습니다

**읽는 State → 갱신할 State:** semantic_query → documents.

준비한 ensemble_retriever를 호출합니다. hybrid에서는 원 질문, combined에서는 분리한 줄거리 입력이 semantic_query에 들어 있습니다.

**작성할 부분:** `docs`에는 `ensemble_retriever.invoke`에 State의 semantic_query를 전달한 결과를 담습니다. 노드는 documents만 갱신합니다.


In [ ]:
def search_hybrid(state: RoutingState):
    """전문·벡터 검색을 RRF로 합친 문서 목록을 반환합니다."""
    # 줄거리 검색어를 전문 검색과 벡터 검색에 공통으로 전달
    docs = ...
    return {"documents": docs}


### 2-4. ④ search_graph: 관계·속성을 조회합니다

**읽는 State → 갱신할 State:** graph_request, route → graph_rows, cypher.

`Neo4jGraph`, 긴 Cypher 프롬프트와 `graph_chain`은 제공합니다. `graph`는 DB 연결 객체이고, 뒤에서 만들 `routing_graph`는 실행 흐름입니다.

graph 모드는 facts·claim_ids를, combined 모드는 source_id까지 반환합니다. `return_direct=True`로 조회 행을 공통 답변 노드에 전달합니다. `allow_dangerous_requests=True`는 생성 쿼리 실행 동의이며 DB 권한을 제한하는 설정은 아닙니다.

**작성할 부분:**

- `result`: graph_chain에 `query=state["graph_request"]`, `search_mode=state["route"]`를 담은 딕셔너리를 전달합니다.
- `update`: result의 result 목록은 graph_rows, intermediate_steps 첫 항목의 query는 cypher에 담습니다.

graph 경로는 질문 임베딩을 호출하지 않습니다. 생성 쿼리를 보관해 원 질문의 조건과 대조합니다.


In [ ]:
from langchain_neo4j import GraphCypherQAChain, Neo4jGraph
from langchain_core.prompts import PromptTemplate

# 이 실습에서 적재한 영화, 인물, 원문, 관계의 스키마를 사용합니다.
graph = Neo4jGraph(
    url=neo4j_uri, username=os.environ["NEO4J_USER"],
    password=os.environ["NEO4J_PASSWORD"], enhanced_schema=False,
)
cypher_prompt = PromptTemplate.from_template("""
주어진 스키마와 질문으로 조회용 Cypher 하나만 반환하세요.
검색 모드: {search_mode}
공통 규칙과 현재 검색 모드의 규칙을 함께 적용하세요.

[공통 규칙]
- 데이터를 변경하지 말고, 스키마의 레이블·속성·관계 방향을 따르세요.
- 질문에 명시된 그래프 대상·관계·속성·숫자·범위 조건을 보존하고, 없는 조건을 추가하지 마세요.
- 개수 조건은 WITH에서 count(DISTINCT 대상) AS 별칭으로 집계한 뒤 WHERE에서 별칭과 기준값을 비교하세요. WHERE에 count()를 직접 쓰지 마세요.
- 이후 절에서 쓸 변수와 facts에 필요한 속성은 각 WITH에 변수 또는 별칭으로 전달하세요.

[공통 반환값]
- facts: 질문에 필요한 이름·제목·속성값·집계값을 담은 맵. 노드·관계 객체나 evidence 원문은 넣지 마세요.
- claim_ids: 답의 근거로 사용한 모든 관계의 실제 claim_id 목록. 중복을 제거하고, 없으면 []를 반환하세요.

[graph 모드]
- 그래프 관계·속성을 조회하고 다음 두 열만 반환하세요.
RETURN facts AS facts, claim_ids AS claim_ids LIMIT 100

[combined 모드]
- 그래프 관계·속성 조건에 맞고 Document 원문이 연결된 후보를 조회하세요.
- 줄거리·사건·분위기는 전문·벡터 검색이 처리합니다. 질문에 포함돼도 Cypher 조건으로 만들지 마세요.
- 줄거리를 Document.text·Chunk.text의 CONTAINS·정규식이나 추측한 그래프 속성으로 필터링하지 마세요.
- Document.id를 source_id로 사용해 원문별 한 행, 다음 세 열만 반환하세요.
RETURN d.id AS source_id, facts AS facts, claim_ids AS claim_ids ORDER BY source_id LIMIT 100

[스키마]
{schema}

[질문]
{question}
""")
graph_chain = GraphCypherQAChain.from_llm(
    llm=llm, graph=graph, cypher_prompt=cypher_prompt,
    include_types=["Movie", "Person", "Document", "ACTED_IN", "DIRECTED",
                   "PRODUCED", "WROTE", "REVIEWED", "FOLLOWS", "ABOUT_MOVIE"],
    return_direct=True, return_intermediate_steps=True, top_k=100,
    # 생성 쿼리 실행에 동의하는 설정이며 DB 쓰기 권한을 제한하지는 않습니다.
    allow_dangerous_requests=True,
)


def search_graph(state: RoutingState):
    """관계·속성을 조회하고 생성 Cypher와 조회 행을 함께 남깁니다."""
    # 조회 행과 생성 Cypher를 함께 보관해 결과 확인에 사용
    result = ...
    update = ...
    return update


### 2-5. ⑤ collect: 답변에 사용할 근거를 모읍니다

**읽는 State → 갱신할 State:** route, documents, graph_rows → source_ids, evidence.

아래 코드는 제공합니다. hybrid는 검색 문서를, graph는 조회 사실과 관계를 사용합니다. combined는 그래프 조건을 통과한 원문을 RRF 순서대로 남기고, 그래프에서만 찾은 후보를 뒤에 보완합니다. 최대 3편의 원문과 관련 관계 근거를 전달합니다.

**그래프 조건 통과는 줄거리 일치의 증명이 아닙니다.** 보완한 후보도 원문을 읽어 확인해야 합니다. 후보 수 제한 때문에 모든 정답을 찾는다고 보장하지 않습니다.


In [ ]:
def collect(state: RoutingState):
    """검색 순위에 그래프 조건을 적용하고 답변 근거를 모읍니다.

    Args:
        state: 경로, 하이브리드 문서, 그래프 조회 행을 가진 상태.
    Returns:
        선택한 원문 ID와 원문·조회 사실·관계 근거.
    """
    if state["route"] == "graph":
        # 관계 조회만 했다면 줄거리 원문 후보를 만들 필요가 없습니다.
        return {"source_ids": [], "evidence": collect_evidence(state["graph_rows"])}

    source_ids = [doc.metadata["source_id"] for doc in state["documents"]]
    selected_graph = []
    if state["route"] == "combined":
        # 그래프 조건을 통과한 원문만 RRF 순서대로 선택
        allowed = {row["source_id"] for row in state["graph_rows"]}
        candidates = [source_id for source_id in source_ids if source_id in allowed]
        # 순위에 없던 그래프 후보를 뒤에 보완합니다. 줄거리 일치는 원문으로 확인합니다.
        candidates += sorted(allowed - set(source_ids))
        source_ids = candidates
    top_ids = source_ids[:3]
    if state["route"] == "combined":
        selected_graph = [row for row in state["graph_rows"] if row["source_id"] in top_ids]
    return {"source_ids": top_ids, "evidence": collect_evidence(selected_graph, top_ids)}


### 2-6. ⑥ answer: 근거를 인용해 답변합니다

**읽는 State → 갱신할 State:** question, evidence → answer, evidence_ids.

출력 스키마·프롬프트·체인·노드를 제공합니다. 원 question과 evidence만으로 답하고, 본문 인용과 evidence_ids를 함께 반환합니다. 세 경로 모두 이 노드를 사용합니다.


In [ ]:
# 세 경로 모두 질문에 필요한 내용과 실제 근거 ID만 반환합니다.
class SearchAnswer(BaseModel):
    """검색 근거에 기반한 답변과 인용 ID 목록입니다."""
    # 본문 인용을 빠뜨리지 않도록 대괄호 ID 형식도 출력 스키마에 지정합니다.
    answer: str = Field(pattern=r"\[[A-Za-z0-9:_-]+\]|근거가 부족합니다", description="각 주장 뒤에 [실제 evidence_id]를 포함한 간결한 한국어 답변")
    evidence_ids: list[str] = Field(description="답변 본문에 인용한 모든 근거 ID를 중복 없이 담은 목록")


search_answer_prompt = ChatPromptTemplate.from_messages([
    ("system", "제공된 근거만 사용해 한국어로 간결하게 답하세요. "
               "각 주장 끝에 실제 근거 ID를 [ID]로 표시하고, 공유 관계는 양쪽 관계 근거를 인용하세요. "
               "ID는 접두어를 포함한 evidence_id 전체를 그대로 복사하고 대괄호 안에 공백을 넣지 마세요. "
               "evidence_ids에는 본문에 쓴 ID만 중복 없이 담으세요. "
               "근거가 없으면 '근거가 부족합니다'라고 답하고 인용 목록을 비우세요."),
    ("human", "질문: {question}\n근거: {context}"),
])
search_answer_chain = search_answer_prompt | llm.with_structured_output(SearchAnswer, strict=True)


def answer(state: RoutingState):
    """원 질문과 선택한 근거로 답하고 인용 ID를 반환합니다."""
    result = search_answer_chain.invoke({
        "question": state["question"], "context": json.dumps(state["evidence"], ensure_ascii=False),
    })
    return {"answer": result.answer, "evidence_ids": result.evidence_ids}


## 3. 엣지를 연결합니다

### 3-1. 라우팅 함수를 만듭니다

라우팅 함수는 다음 경로의 이름을 반환합니다. **노드처럼 State를 갱신하거나 Jev를 다시 호출하지 않습니다.**

| 그림의 위치 | 함수 | 반환값 -> 다음 노드 |
|---|---|---|
| **분기 A · ① 뒤** | route_search | hybrid -> ③, graph -> ④, combined -> ② |
| **분기 B · ③ 뒤** | after_hybrid | graph -> ④, collect -> ⑤ |

**A:** selected_route에 State의 route를 넣습니다. **B:** route가 combined이면 next_step은 graph, 그 외에는 collect입니다. 그림은 분기 조건을, 연결표는 함수의 반환값을 표시합니다.


In [ ]:
def route_search(state: RoutingState):
    """저장된 경로를 읽습니다. Jev를 다시 부르지 않습니다."""
    # 분기 A: ①에서 저장한 경로로 ②, ③, ④ 중 다음 노드 선택
    selected_route = ...
    return selected_route


def after_hybrid(state: RoutingState):
    """복합 질문이면 그래프 조회를 추가하고, 그 외에는 근거를 모읍니다."""
    # 분기 B: ③ 뒤에서 combined면 ④ 관계 조회, 그 외에는 ⑤ 근거 수집
    if state["route"] == "combined":
        next_step = ...
    else:
        next_step = ...
    return next_step


### 3-2. StateGraph에 노드를 등록합니다

`builder`에는 RoutingState를 사용하는 StateGraph를 만듭니다. add_node 호출은 제공하며, 이름과 실행 함수의 대응을 확인하세요.


In [ ]:
from langgraph.graph import END, START, StateGraph

# 그래프 구성 객체 생성 및 노드 등록: 이름과 실행 함수 지정
builder = ...
builder.add_node("classify", classify)
builder.add_node("split_query", split_query)
builder.add_node("search_hybrid", search_hybrid)
builder.add_node("search_graph", search_graph)
builder.add_node("collect", collect)
builder.add_node("answer", answer)


### 3-3. 실행 순서와 두 분기를 연결합니다


<img src="images/12_lesson03_paths.png" width="1100" style="background:white;display:block;margin:16px auto" alt="hybrid·graph·combined가 공통 노드를 재사용하며 필요한 검색만 실행하는 세 경로">



그림의 같은 이름은 같은 노드를 경로별로 반복 표시한 것입니다. combined의 두 검색은 순서대로 실행합니다.

1. START를 classify에 연결합니다.
2. classify 뒤에 route_search와 첫 연결표를 등록합니다.
3. split_query를 search_hybrid에 연결합니다. search_hybrid 뒤에는 after_hybrid와 두 번째 연결표를 등록합니다.
4. search_graph -> collect -> answer -> END를 연결합니다.
5. `routing_graph`에는 builder를 compile한 실행 객체를 담습니다.

분기 위치에는 조건부 엣지를 사용합니다. 고정 엣지를 함께 붙이면 원하지 않는 경로도 실행될 수 있습니다. 연결을 바꿀 때는 **새 builder 생성·등록부터** 다시 실행하세요. [조건부 엣지 공식 문서](https://docs.langchain.com/oss/python/langgraph/graph-api#conditional-edges)


In [ ]:
# (1) 입구를 classify에 연결하세요.
...
# (2) classify 뒤의 세 경로를 route_search로 연결하세요.
...
# (3) split_query 뒤와 search_hybrid 뒤를 연결하세요.
...
# (4) search_graph 뒤에 collect, answer, END를 차례로 연결하세요.
...
# (5) 실행 가능한 그래프를 만드세요.
routing_graph = ...


### 연결 결과를 이미지로 확인합니다

맨 위 개요도와 같은 노드·분기인지 비교하세요. 이 그림은 실제 compile 결과이며 점선은 조건부 연결입니다. 이 셀은 그래프를 실행하지 않으며, 이미지 렌더링에 네트워크 연결이 필요할 수 있습니다.


In [ ]:
# 컴파일한 그래프의 노드와 엣지 표시
from IPython.display import Image, display
display(Image(routing_graph.get_graph().draw_mermaid_png()))


## 4. 그래프를 실행하고 결과를 확인합니다

`new_result`에는 routing_graph.invoke의 최종 State를 담습니다. 입력은 initial_state를 펼치고 question을 new_question으로 바꾼 딕셔너리입니다.

**실제 DB·API를 호출합니다.** 예시가 combined로 선택되고 한글 검색어 변환을 거치면 Jev는 경로·언어 판단 2회, GPT는 질문 분리·검색어 변환·Cypher·답변 4회, 질문 임베딩은 1회 호출합니다. 문서 임베딩은 호출하지 않습니다.


In [ ]:
# 초기 상태에 새 질문을 넣어 그래프 실행
new_result = ...


### 선택 경로와 검색 결과를 읽습니다

combined가 선택됐는지, 분리한 graph_request와 Cypher에 **공유 배우 최소 2명·1993년 제외·기준 영화 제외**가 남았는지 확인합니다. 배우 중복은 DISTINCT 집계로 제거해야 합니다.


In [ ]:
# 저장된 실행 결과에서 경로, 검색 입력, Cypher, 후보를 출력합니다.
print("선택 경로:", new_result["route"])
if new_result["route"] == "combined":
    print("줄거리 검색 입력:", new_result["semantic_query"])
    print("그래프 조회 입력:", new_result["graph_request"])
print("생성 Cypher:\n", new_result["cypher"])
display(pd.DataFrame(new_result["graph_rows"]))
display(pd.DataFrame([
    {"source_id": source_id, "title": documents_by_id[source_id]["title"]}
    for source_id in new_result["source_ids"]
]))


### 답변의 인용을 원문과 대조합니다

후보 원문이 경쟁 서점·익명 이메일 조건을 뒷받침하는지, 본문의 인용 ID가 제공 근거에 있는지 확인합니다. 확인 기준은 You've Got Mail과 Tom Hanks·Meg Ryan의 출연 관계입니다. Joe Versus the Volcano는 연도 조건을 만족하지 않습니다.


In [ ]:
print("최종 답변:")
display(Markdown(new_result["answer"]))
print("인용 ID:", new_result["evidence_ids"])

# 인용 ID 대조와 원문 출력: 답변이 원문과 일치하는지는 직접 확인
evidence_by_id = {item["evidence_id"]: item for item in new_result["evidence"]}
inline_ids = set(re.findall(r"\[([^\[\]]+)\]", new_result["answer"]))
unknown_ids = (inline_ids | set(new_result["evidence_ids"])) - set(evidence_by_id)
print("제공하지 않은 인용 ID:", unknown_ids)
print("본문과 인용 목록의 차이:", inline_ids ^ set(new_result["evidence_ids"]))
for evidence_id in new_result["evidence_ids"]:
    if evidence_id in evidence_by_id:
        print("\n근거:", evidence_id)
        print(evidence_by_id[evidence_id]["text"])


### 흐름을 설명하며 복습합니다

- 영화 제목이 있어도 hybrid를 고르는 질문을 하나 만드세요.
- graph 경로에서 documents가 비는 이유를 연결 구조로 설명하세요.
- combined에서 하이브리드 검색 뒤 바로 답하면 어떤 조건을 놓치나요?
- 오분류는 분류 기준, 조건 누락은 질문 분리·Cypher, 근거 밖 답변은 원문·답변 프롬프트 중 어디부터 확인해야 하나요?

## 교안 03 핵심 코드 이어서 보기

**새 커널**에서 아래 영역만 실행할 수 있습니다. 본문과 같은 코드이며 학생용에도 완성본을 제공합니다. 데이터 적재부터 반복하고 마지막 질문에서 실제 API를 호출하므로, 복습에 필요할 때만 실행하세요.

### 준비. 데이터와 공통 검색 도구


In [ ]:
# 모델과 DB 연결에 필요한 라이브러리를 불러옵니다.
import json
import os
import re
import sys
from pathlib import Path
from IPython.display import Markdown, display
from urllib.parse import urlparse

import pandas as pd
from dotenv import find_dotenv, load_dotenv
from neo4j import GraphDatabase
from langchain_openai import ChatOpenAI, OpenAIEmbeddings



# 노트북 기준 경로입니다. 정답은 상위 폴더를 사용합니다.
material_dir = Path(".")
data_dir = material_dir / "data"
sys.path.insert(0, str(material_dir.resolve()))
from graph_data import load_graph, read_vectors


# 공통 설정을 읽고 교안의 .env를 우선 적용합니다.
load_dotenv(find_dotenv(usecwd=True))
load_dotenv(material_dir.resolve().parent / ".env")
load_dotenv(material_dir / ".env", override=True)
# 모델만 준비합니다. API 호출은 검색하거나 답변할 때 합니다.
embedding_model = OpenAIEmbeddings(
    model="text-embedding-3-large", dimensions=768,
    check_embedding_ctx_length=False,  # 준비한 청크를 자동으로 나누지 않습니다.
)
llm = ChatOpenAI(model=os.getenv("OPENAI_CHAT_MODEL", "gpt-5.6-luna"))

# .env의 주소와 계정으로 Neo4j에 연결합니다.
neo4j_uri = os.environ["NEO4J_URI"]
driver = GraphDatabase.driver(
    neo4j_uri, auth=(os.environ["NEO4J_USER"], os.environ["NEO4J_PASSWORD"])
)
driver.verify_connectivity()
address = urlparse(neo4j_uri)
print("Neo4j 연결:", address.hostname, address.port or 7687)


def run_cypher(query, **params):
    """Cypher 결과를 행 딕셔너리의 리스트로 반환합니다."""
    # 인자를 Cypher의 $파라미터로 전달합니다.
    with driver.session() as session:
        # 조회 결과를 딕셔너리 목록으로 반환합니다.
        return [record.data() for record in session.run(query, **params)]


# 영화 그래프와 원문, 기존 청크 벡터를 불러옵니다.
movies = load_graph(data_dir / "movies_extraction_packet.json")
chunk_ids, vectors = read_vectors(
    data_dir / "embeddings/movies_complete.npz", movies["chunks"], embedding_model
)
# 검색 결과의 원문 ID로 줄거리를 찾을 딕셔너리입니다.
documents_by_id = movies["documents"]
print("개체:", len(movies["nodes"]), "관계:", len(movies["relations"]))
print("문서:", len(documents_by_id), "청크:", len(chunk_ids), "벡터:", vectors.shape)


# 이전 실습의 공용 Chunk.embedding 벡터 인덱스만 정리합니다.
# 노드, 관계, 저장된 임베딩은 유지됩니다.
indexes = run_cypher("""
    SHOW VECTOR INDEXES
    YIELD name, labelsOrTypes, properties
    WHERE labelsOrTypes = ['Chunk'] AND properties = ['embedding']
    RETURN name
""")

for index in indexes:
    name = index["name"].replace("`", "``")
    run_cypher(f"DROP INDEX `{name}` IF EXISTS")
    print("삭제:", index["name"])

# 개체의 standard_id 중복을 막는 고유 제약을 만듭니다.
run_cypher("CREATE CONSTRAINT rag_entity_id IF NOT EXISTS FOR (n:RAGEntity) REQUIRE n.standard_id IS UNIQUE")
# 원문 id 중복을 막는 고유 제약을 만듭니다.
run_cypher("CREATE CONSTRAINT document_id IF NOT EXISTS FOR (n:Document) REQUIRE n.id IS UNIQUE")
# 청크 id 중복을 막는 고유 제약을 만듭니다.
run_cypher("CREATE CONSTRAINT chunk_id IF NOT EXISTS FOR (n:Chunk) REQUIRE n.id IS UNIQUE")

# 영화와 인물을 종류별로 적재합니다.
for entity_type in ["Movie", "Person"]:
    # 현재 종류의 개체만 고릅니다.
    rows = [row for row in movies["nodes"] if row["entity_type"] == entity_type]
    # 현재 종류의 영화 또는 인물 노드를 만들고 속성을 갱신합니다.
    run_cypher(f"""
        // 목록을 개체별 행으로 펼칩니다.
        UNWIND $rows AS item
        // 같은 ID의 노드를 재사용하고, 없으면 생성합니다.
        MERGE (n:RAGEntity {{standard_id: item.standard_id}})
        // 종류 레이블을 추가하고 속성·이름을 갱신합니다.
        SET n:{entity_type}, n += item.properties, n.name = item.name
    """, rows=rows)

# 적재한 노드 사이에 종류별 관계를 연결합니다.
for relation in ["ACTED_IN", "DIRECTED", "PRODUCED", "WROTE", "REVIEWED", "FOLLOWS"]:
    # 현재 유형의 관계만 고릅니다.
    rows = [row for row in movies["relations"] if row["relation"] == relation]
    # 현재 유형의 관계를 연결하고 근거와 출처를 저장합니다.
    run_cypher(f"""
        // 목록을 관계별 행으로 펼칩니다.
        UNWIND $rows AS item
        // 관계의 시작·끝 노드를 ID로 찾습니다.
        MATCH (a:RAGEntity {{standard_id: item.subject_id}})
        MATCH (b:RAGEntity {{standard_id: item.object_id}})
        // 같은 유형·방향·근거 ID의 관계가 없으면 만들고, 있으면 재사용합니다.
        MERGE (a)-[r:{relation} {{claim_id: item.claim_id}}]->(b)
        // 관계 속성과 근거·출처를 저장합니다.
        SET r += item.properties, r.evidence = item.evidence,
            r.source_url = item.source_url, r.source_doc_id = item.source_doc_id
    """, rows=rows)


# 원문 노드를 먼저 저장합니다.
run_cypher("""
    // 원문 목록을 한 문서당 한 행으로 펼칩니다.
    UNWIND $rows AS item
    // 같은 ID의 원문 노드가 없으면 만들고, 있으면 재사용합니다.
    MERGE (d:Document {id: item.doc_id})
    // 제목·본문·출처 URL을 갱신합니다.
    SET d.title = item.title, d.text = item.text, d.url = item.url
""", rows=list(documents_by_id.values()))

# 원문과 해당 영화를 연결합니다.
run_cypher("""
    // doc_id로 원문을, standard_id로 해당 영화를 찾습니다.
    UNWIND $rows AS item
    MATCH (d:Document {id: item.doc_id})
    MATCH (m:Movie {standard_id: item.standard_id})
    // 원문과 해당 영화를 ABOUT_MOVIE 관계로 연결합니다.
    MERGE (d)-[:ABOUT_MOVIE]->(m)
""", rows=movies["document_links"])

# 청크와 벡터의 순서를 맞춰 전체 저장 목록을 만듭니다.
rows = []
for index, chunk in enumerate(movies["chunks"]):
    # NumPy 벡터를 DB에 전달할 목록으로 바꿉니다.
    rows.append({"id": chunk_ids[index], "doc_id": chunk.metadata["source_doc_id"],
                 "text": chunk.page_content, "embedding": vectors[index].tolist()})

# 전체 청크와 벡터를 한 번에 저장합니다.
run_cypher("""
    UNWIND $rows AS item
    // 원문을 찾고 청크를 ID로 생성·재사용합니다.
    MATCH (d:Document {id: item.doc_id})
    MERGE (c:Chunk {id: item.id})
    // 이 교안의 영화 청크만 검색하도록 전용 레이블을 붙입니다.
    SET c:Day49MovieChunk
    // text는 전문 검색, embedding은 벡터 검색, source_id는 원문별 결과 통합에 씁니다.
    SET c.text = item.text, c.source_id = item.doc_id, c.embedding = item.embedding
    // 청크와 원문을 FROM_DOCUMENT 관계로 연결합니다.
    MERGE (c)-[:FROM_DOCUMENT]->(d)
""", rows=rows)


# 같은 이름의 옛 설정과 같은 검색 대상의 다른 이름 인덱스를 정리합니다.
# IF NOT EXISTS만 쓰면 다른 이름의 인덱스 때문에 필요한 이름이 생성되지 않을 수 있습니다.
indexes = run_cypher("""
    SHOW INDEXES YIELD name, type, entityType, labelsOrTypes, properties
    WHERE name IN ['movie_chunks', 'movie_fulltext']
       OR (entityType = 'NODE' AND labelsOrTypes = ['Day49MovieChunk']
           AND ((type = 'VECTOR' AND properties = ['embedding'])
             OR (type = 'FULLTEXT' AND properties = ['text'])))
    RETURN name
""")
for index in indexes:
    name = index["name"].replace("`", "``")
    run_cypher(f"DROP INDEX `{name}` IF EXISTS")
# 아래에서 정해진 이름, 차원, 분석기로 다시 만듭니다. 노드와 저장된 벡터는 유지됩니다.

# 벡터 유사도 검색용 인덱스를 만듭니다.
run_cypher("""
    CREATE VECTOR INDEX movie_chunks IF NOT EXISTS
    // 이 교안의 영화 청크만 벡터 검색 대상으로 지정합니다.
    FOR (c:Day49MovieChunk) ON c.embedding
    // 저장된 벡터의 768차원과 코사인 유사도를 설정합니다.
    OPTIONS {indexConfig: {`vector.dimensions`: 768, `vector.similarity_function`: 'cosine'}}
""")
# 벡터 인덱스가 검색 가능한 상태가 될 때까지 최대 300초 기다립니다.
run_cypher("CALL db.awaitIndex('movie_chunks', 300)")
# 단어 일치 검색용 전문 인덱스를 만듭니다.
run_cypher("""
    CREATE FULLTEXT INDEX movie_fulltext IF NOT EXISTS
    // 같은 영화 청크의 text만 전문 검색 대상으로 지정합니다.
    FOR (c:Day49MovieChunk) ON EACH [c.text]
    // 영어 분석기를 사용하고, 본문 변경을 커밋 시 인덱스에 반영합니다.
    OPTIONS {indexConfig: {`fulltext.analyzer`: 'english', `fulltext.eventually_consistent`: false}}
""")
# 전문 인덱스가 검색 가능한 상태가 될 때까지 최대 300초 기다립니다.
run_cypher("CALL db.awaitIndex('movie_fulltext', 300)")
print("기존 문서 임베딩을 적재했습니다. 문서 임베딩 API 호출은 없습니다.")


#### 공통 검색 도구


In [ ]:
from langchain_typesafe import Choice, TypeSafeClassifier
from neo4j_graphrag.retrievers import VectorRetriever
from neo4j_graphrag.types import RetrieverResultItem
from langchain_core.documents import Document
from langchain_core.runnables import RunnableLambda
from langchain_classic.retrievers import EnsembleRetriever
from pydantic import BaseModel, Field
from langchain_core.prompts import ChatPromptTemplate

def format_vector(record):
    """검색된 청크의 본문과 원문 ID·유사도 점수를 반환합니다."""
    node = record["node"]
    # content는 청크 본문, metadata는 청크 ID, 원문 ID, 유사도를 담는 딕셔너리입니다.
    return RetrieverResultItem(content=node["text"], metadata={
        "chunk_id": node["id"], "source_id": node["source_id"], "score": record["score"]
    })


# 벡터 인덱스와 결과 변환 함수를 연결해 벡터 검색기를 만듭니다.
vector_retriever = VectorRetriever(
    driver, index_name="movie_chunks", embedder=embedding_model,
    result_formatter=format_vector,
)


def vector_documents(query_vector, top_k=12):
    """검색된 청크 중 영화별 최고 점수 한 건을 대표 후보로 반환합니다."""
    # 질문 벡터와 유사한 청크를 점수 내림차순으로 검색합니다.
    result = vector_retriever.search(query_vector=query_vector, top_k=top_k)
    # 영화별 후보 목록을 만들기 위해 처음 나온 최고 점수 청크를 대표로 남깁니다.
    rows, seen = [], set()
    for item in result.items:
        row = item.metadata
        if row["source_id"] not in seen:
            seen.add(row["source_id"])
            rows.append(dict(row))
    return rows


# Jev가 한글과 영어 중 질문의 언어를 선택하도록 설정합니다.
language_classifier = TypeSafeClassifier(model=os.getenv("TYPESAFE_MODEL", "jev-1.13.0"))
language_question = Choice(
    instructions="질문의 주된 언어를 선택하세요.",
    criteria={"ko": "한글 질문. 영어 영화명 등이 섞인 한글 문장도 포함합니다.",
              "en": "한글 없이 영어 단어나 문장으로 작성된 질문입니다."},
)


# 영어 검색어를 문자열 목록으로 받습니다.
class EnglishKeywords(BaseModel):
    """영어 원문 검색에 사용할 핵심어 목록입니다."""
    keywords: list[str] = Field(description="중복 없는 영어 단어 4~8개")


# 한글 질문에서 영어 검색어 목록을 만드는 체인입니다.
lexical_prompt = ChatPromptTemplate.from_messages([
    ("system", "한글 영화 줄거리 질문을 영어 원문 검색용 핵심어 4~8개로 바꾸세요. "
               "질문의 핵심 사건·대상을 나타내는 단어를 골라 keywords의 각 항목에 영어 단어 하나씩 담으세요. "
               "영화 정답이나 배우를 추측해 추가하지 마세요. 질문에 없는 고유명사는 넣지 마세요."),
    ("human", "{question}"),
])
lexical_chain = lexical_prompt | llm.with_structured_output(EnglishKeywords)


def fulltext_documents(query, top_k=12):
    """질문의 언어에 맞춰 전문 검색하고 원문별 최고 점수 청크를 반환합니다."""
    # state에 질문, questions에 언어 선택 기준을 전달합니다.
    decision = language_classifier.invoke({
        "state": query, "questions": {"language": language_question},
    })
    language = decision.choices["language"].choice
    # 한글은 영어 검색어 목록을 받아 OR로 연결하고, 영어는 그대로 사용합니다.
    if language == "ko":
        terms = lexical_chain.invoke({"question": query})
        search_text = " OR ".join(f'"{word}"' for word in terms.keywords)
    else:
        search_text = query
    print("질문 언어:", language, "| 전문 검색어:", search_text)
    # 전문 인덱스에서 검색어와 일치하는 청크를 최대 top_k개 조회합니다.
    hits = run_cypher("""
        // 전문 인덱스에서 관련 청크를 최대 top_k개 조회합니다.
        CALL db.index.fulltext.queryNodes('movie_fulltext', $search_text, {limit: $top_k})
        YIELD node, score
        // 결과 통합에 쓸 원문 ID를 반환합니다.
        RETURN node.source_id AS source_id, node.id AS chunk_id, score
        // 점수 내림차순으로 정렬합니다.
        ORDER BY score DESC, chunk_id
    """, search_text=search_text, top_k=top_k)
    rows, seen = [], set()
    # 영화별 순위를 비교하도록 원문마다 최고 점수 검색 행을 대표로 남깁니다.
    for row in hits:
        source_id = row["source_id"]
        if source_id not in seen:
            seen.add(source_id)
            rows.append(row)
    return rows




def as_documents(rows):
    """원문별 검색 결과를 LangChain Document 목록으로 바꿉니다."""
    return [Document(page_content=documents_by_id[row["source_id"]]["text"],
                     metadata={"source_id": row["source_id"]}) for row in rows]


# 기존 검색 함수를 LangChain에서 호출할 수 있도록 연결합니다.
# 두 함수는 이미 원문별 중복을 제거한 결과를 반환합니다.
fulltext_retriever = RunnableLambda(
    lambda query: as_documents(fulltext_documents(query, top_k=30))
)
dense_retriever = RunnableLambda(
    lambda query: as_documents(vector_documents(embedding_model.embed_query(query), top_k=30))
)
# 전문 검색과 벡터 검색을 가중 RRF로 결합합니다. 같은 원문은 source_id로 합칩니다.
ensemble_retriever = EnsembleRetriever(
    retrievers=[fulltext_retriever, dense_retriever],
    weights=[0.4, 0.6], c=60, id_key="source_id",
)

# 실제 저장된 관계 ID로 근거 문장을 찾습니다.
relations_by_id = {row["claim_id"]: row for row in movies["relations"]}


def collect_evidence(graph_rows, source_ids=()):
    """선택한 원문·그래프 조회 사실·관계 근거를 하나의 목록으로 모읍니다."""
    evidence = []
    for source_id in source_ids:
        doc = documents_by_id[source_id]
        evidence.append({"evidence_id": source_id, "kind": "plot", "title": doc["title"],
                         "url": doc["url"], "text": doc["text"]})
    claim_ids = set()
    for i, row in enumerate(graph_rows, start=1):
        evidence.append({"evidence_id": f"graph:{i}", "kind": "graph_record",
                         "text": json.dumps(row["facts"], ensure_ascii=False)})
        claim_ids.update(row["claim_ids"])
    for claim_id in sorted(claim_ids):
        evidence.append({"evidence_id": claim_id, "kind": "relation",
                         "text": relations_by_id[claim_id]["evidence"]})
    return evidence


### 1. State와 입력 생성


In [ ]:
from typing import TypedDict

# 검색 워크플로의 상태 스키마 정의
class RoutingState(TypedDict):
    question: str  # 검색 경로 분류와 최종 답변에 사용할 원 질문
    route: str  # 필요한 검색을 연결할 경로: hybrid, graph, combined
    semantic_query: str  # 전문 검색과 벡터 검색에 전달할 줄거리 검색어
    graph_request: str  # Cypher 생성에 전달할 관계와 속성 조건
    documents: list[Document]  # 답변에 쓸 원문 후보를 고르기 위한 하이브리드 검색 결과
    graph_rows: list[dict]  # 근거 수집에 쓸 조회 결과: facts(사실), claim_ids(관계 ID 목록)
    cypher: str  # 질문의 조건이 반영됐는지 확인할 생성 쿼리
    source_ids: list[str]  # 원문 근거를 불러올 때 사용할 선택된 문서 ID
    evidence: list[dict]  # GPT가 읽을 근거: evidence_id(인용 ID), kind(근거 유형), text(내용)
    answer: str  # 사용자에게 보여 줄 근거 기반 최종 답변
    evidence_ids: list[str]  # 답변의 인용을 실제 근거와 대조할 때 사용할 ID


# 초기 상태 선언: 키는 State 필드 이름, 값은 해당 필드의 시작 데이터
initial_state: RoutingState = {
    "question": "", "route": "",
    "semantic_query": "", "graph_request": "", "documents": [],
    "graph_rows": [], "cypher": "", "source_ids": [], "evidence": [],
    "answer": "", "evidence_ids": [],
}
new_question = (
    "Sleepless in Seattle과 출연 배우를 최소 2명 공유하고 "
    "1993년 이후(1993년 제외) 개봉한 다른 영화 중 "
    "서로 경쟁하는 서점 주인들이 익명 이메일로 사랑하게 되는 작품을 찾아 "
    "공통 배우와 근거를 보여 주세요."
)


### 2. 노드 생성

#### 2-1. 분류 노드


In [ ]:
# 질문 유형 판단도 기존 언어 판단과 같은 Jev 모델을 사용합니다.
route_classifier = TypeSafeClassifier(model=os.getenv("TYPESAFE_MODEL", "jev-1.13.0"))
route_question = Choice(
    instructions="영화 질문에 필요한 검색 경로를 선택하세요. 영화 제목 언급만으로 graph를 선택하지 마세요.",
    criteria={
        "hybrid": "배우 관계나 개봉연도 조건 없이 줄거리·사건·분위기로 영화를 찾거나 원문 내용을 묻는 질문",
        "graph": "줄거리 조건 없이 배우의 출연·공유 배우·개봉연도 등 그래프 관계나 속성만 조회하는 질문",
        "combined": "줄거리·사건·분위기 조건과 배우 관계·개봉연도 등 그래프 조건을 함께 만족해야 하는 질문",
    },
)


def classify(state: RoutingState):
    """Jev가 고른 경로와 두 검색의 기본 입력을 준비합니다."""
    question = state["question"].strip()
    decision = route_classifier.invoke({"state": question, "questions": {"route": route_question}})
    # 두 검색 입력을 원 질문으로 초기화: combined 경로에서는 split_query가 갱신
    update = {
        "question": question, "route": decision.choices["route"].choice,
        "semantic_query": question, "graph_request": question,
    }
    return update


#### 2-2. 질문 분리 노드


In [ ]:
# 검색별 입력을 정해진 두 필드로 받습니다.
class SearchQueries(BaseModel):
    """원 질문에서 분리한 줄거리 검색어와 그래프 조건 질문입니다."""
    semantic_query: str = Field(description="전문·벡터 검색에 사용할 줄거리의 사건·상황")
    graph_request: str = Field(description="질문에 명시된 모든 그래프 관계·속성 조건을 담은 자연어 질문")


# 줄거리는 내용 검색에, 관계와 속성 조건은 그래프 검색에 전달합니다.
query_split_prompt = ChatPromptTemplate.from_messages([
    ("system", "영화 질문을 두 검색 입력으로 나누세요. "
               "semantic_query에는 줄거리의 사건·상황만 담고 그래프 관계·속성 조건은 빼세요. "
               "graph_request에는 질문에 명시된 모든 그래프 관계·속성 조건과 조회 대상을 보존하세요. 예를 들어 출연·감독·제작 관계, 개봉연도, 인원 수, 제외 대상입니다. "
               "graph_request에는 줄거리 조건을 넣지 말고 Cypher가 아닌 자연어 질문을 작성하세요. "
               "숫자와 이상·초과·이전·이후 등 범위 표현을 바꾸지 마세요. "
               "원 질문에 없는 조건·영화명·인물명을 추가하지 마세요."),
    ("human", "{question}"),
])
query_split_chain = query_split_prompt | llm.with_structured_output(SearchQueries)


def split_query(state: RoutingState):
    """복합 질문의 줄거리와 그래프 조건을 분리합니다."""
    # 두 검색기가 각각 사용할 입력으로 분리
    queries = query_split_chain.invoke({"question": state["question"]})
    update = {"semantic_query": queries.semantic_query, "graph_request": queries.graph_request}
    return update


#### 2-3. 하이브리드 검색 노드


In [ ]:
def search_hybrid(state: RoutingState):
    """전문·벡터 검색을 RRF로 합친 문서 목록을 반환합니다."""
    # 줄거리 검색어를 전문 검색과 벡터 검색에 공통으로 전달
    docs = ensemble_retriever.invoke(state["semantic_query"])
    return {"documents": docs}


#### 2-4. 관계 검색 노드


In [ ]:
from langchain_neo4j import GraphCypherQAChain, Neo4jGraph
from langchain_core.prompts import PromptTemplate

# 이 실습에서 적재한 영화, 인물, 원문, 관계의 스키마를 사용합니다.
graph = Neo4jGraph(
    url=neo4j_uri, username=os.environ["NEO4J_USER"],
    password=os.environ["NEO4J_PASSWORD"], enhanced_schema=False,
)
cypher_prompt = PromptTemplate.from_template("""
주어진 스키마와 질문으로 조회용 Cypher 하나만 반환하세요.
검색 모드: {search_mode}
공통 규칙과 현재 검색 모드의 규칙을 함께 적용하세요.

[공통 규칙]
- 데이터를 변경하지 말고, 스키마의 레이블·속성·관계 방향을 따르세요.
- 질문에 명시된 그래프 대상·관계·속성·숫자·범위 조건을 보존하고, 없는 조건을 추가하지 마세요.
- 개수 조건은 WITH에서 count(DISTINCT 대상) AS 별칭으로 집계한 뒤 WHERE에서 별칭과 기준값을 비교하세요. WHERE에 count()를 직접 쓰지 마세요.
- 이후 절에서 쓸 변수와 facts에 필요한 속성은 각 WITH에 변수 또는 별칭으로 전달하세요.

[공통 반환값]
- facts: 질문에 필요한 이름·제목·속성값·집계값을 담은 맵. 노드·관계 객체나 evidence 원문은 넣지 마세요.
- claim_ids: 답의 근거로 사용한 모든 관계의 실제 claim_id 목록. 중복을 제거하고, 없으면 []를 반환하세요.

[graph 모드]
- 그래프 관계·속성을 조회하고 다음 두 열만 반환하세요.
RETURN facts AS facts, claim_ids AS claim_ids LIMIT 100

[combined 모드]
- 그래프 관계·속성 조건에 맞고 Document 원문이 연결된 후보를 조회하세요.
- 줄거리·사건·분위기는 전문·벡터 검색이 처리합니다. 질문에 포함돼도 Cypher 조건으로 만들지 마세요.
- 줄거리를 Document.text·Chunk.text의 CONTAINS·정규식이나 추측한 그래프 속성으로 필터링하지 마세요.
- Document.id를 source_id로 사용해 원문별 한 행, 다음 세 열만 반환하세요.
RETURN d.id AS source_id, facts AS facts, claim_ids AS claim_ids ORDER BY source_id LIMIT 100

[스키마]
{schema}

[질문]
{question}
""")
graph_chain = GraphCypherQAChain.from_llm(
    llm=llm, graph=graph, cypher_prompt=cypher_prompt,
    include_types=["Movie", "Person", "Document", "ACTED_IN", "DIRECTED",
                   "PRODUCED", "WROTE", "REVIEWED", "FOLLOWS", "ABOUT_MOVIE"],
    return_direct=True, return_intermediate_steps=True, top_k=100,
    # 생성 쿼리 실행에 동의하는 설정이며 DB 쓰기 권한을 제한하지는 않습니다.
    allow_dangerous_requests=True,
)


def search_graph(state: RoutingState):
    """관계·속성을 조회하고 생성 Cypher와 조회 행을 함께 남깁니다."""
    # 조회 행과 생성 Cypher를 함께 보관해 결과 확인에 사용
    result = graph_chain.invoke({"query": state["graph_request"], "search_mode": state["route"]})
    update = {"graph_rows": result["result"], "cypher": result["intermediate_steps"][0]["query"]}
    return update


#### 2-5. 근거 수집 노드


In [ ]:
def collect(state: RoutingState):
    """검색 순위에 그래프 조건을 적용하고 답변 근거를 모읍니다.

    Args:
        state: 경로, 하이브리드 문서, 그래프 조회 행을 가진 상태.
    Returns:
        선택한 원문 ID와 원문·조회 사실·관계 근거.
    """
    if state["route"] == "graph":
        # 관계 조회만 했다면 줄거리 원문 후보를 만들 필요가 없습니다.
        return {"source_ids": [], "evidence": collect_evidence(state["graph_rows"])}

    source_ids = [doc.metadata["source_id"] for doc in state["documents"]]
    selected_graph = []
    if state["route"] == "combined":
        # 그래프 조건을 통과한 원문만 RRF 순서대로 선택
        allowed = {row["source_id"] for row in state["graph_rows"]}
        candidates = [source_id for source_id in source_ids if source_id in allowed]
        # 순위에 없던 그래프 후보를 뒤에 보완합니다. 줄거리 일치는 원문으로 확인합니다.
        candidates += sorted(allowed - set(source_ids))
        source_ids = candidates
    top_ids = source_ids[:3]
    if state["route"] == "combined":
        selected_graph = [row for row in state["graph_rows"] if row["source_id"] in top_ids]
    return {"source_ids": top_ids, "evidence": collect_evidence(selected_graph, top_ids)}


#### 2-6. 답변 노드


In [ ]:
# 세 경로 모두 질문에 필요한 내용과 실제 근거 ID만 반환합니다.
class SearchAnswer(BaseModel):
    """검색 근거에 기반한 답변과 인용 ID 목록입니다."""
    # 본문 인용을 빠뜨리지 않도록 대괄호 ID 형식도 출력 스키마에 지정합니다.
    answer: str = Field(pattern=r"\[[A-Za-z0-9:_-]+\]|근거가 부족합니다", description="각 주장 뒤에 [실제 evidence_id]를 포함한 간결한 한국어 답변")
    evidence_ids: list[str] = Field(description="답변 본문에 인용한 모든 근거 ID를 중복 없이 담은 목록")


search_answer_prompt = ChatPromptTemplate.from_messages([
    ("system", "제공된 근거만 사용해 한국어로 간결하게 답하세요. "
               "각 주장 끝에 실제 근거 ID를 [ID]로 표시하고, 공유 관계는 양쪽 관계 근거를 인용하세요. "
               "ID는 접두어를 포함한 evidence_id 전체를 그대로 복사하고 대괄호 안에 공백을 넣지 마세요. "
               "evidence_ids에는 본문에 쓴 ID만 중복 없이 담으세요. "
               "근거가 없으면 '근거가 부족합니다'라고 답하고 인용 목록을 비우세요."),
    ("human", "질문: {question}\n근거: {context}"),
])
search_answer_chain = search_answer_prompt | llm.with_structured_output(SearchAnswer, strict=True)


def answer(state: RoutingState):
    """원 질문과 선택한 근거로 답하고 인용 ID를 반환합니다."""
    result = search_answer_chain.invoke({
        "question": state["question"], "context": json.dumps(state["evidence"], ensure_ascii=False),
    })
    return {"answer": result.answer, "evidence_ids": result.evidence_ids}


### 3. 라우팅 함수와 엣지 연결


In [ ]:
def route_search(state: RoutingState):
    """저장된 경로를 읽습니다. Jev를 다시 부르지 않습니다."""
    # 분기 A: ①에서 저장한 경로로 ②, ③, ④ 중 다음 노드 선택
    selected_route = state["route"]
    return selected_route


def after_hybrid(state: RoutingState):
    """복합 질문이면 그래프 조회를 추가하고, 그 외에는 근거를 모읍니다."""
    # 분기 B: ③ 뒤에서 combined면 ④ 관계 조회, 그 외에는 ⑤ 근거 수집
    if state["route"] == "combined":
        next_step = "graph"
    else:
        next_step = "collect"
    return next_step


from langgraph.graph import END, START, StateGraph

# 그래프 구성 객체 생성 및 노드 등록: 이름과 실행 함수 지정
builder = StateGraph(RoutingState)
builder.add_node("classify", classify)
builder.add_node("split_query", split_query)
builder.add_node("search_hybrid", search_hybrid)
builder.add_node("search_graph", search_graph)
builder.add_node("collect", collect)
builder.add_node("answer", answer)


# 분기 A: ① 질문 분류 뒤에서 첫 검색 경로 선택
builder.add_edge(START, "classify")
builder.add_conditional_edges(
    "classify", route_search,
    {"hybrid": "search_hybrid", "graph": "search_graph", "combined": "split_query"},
)
builder.add_edge("split_query", "search_hybrid")
# 분기 B: ③ 하이브리드 검색 뒤에서 ④ 관계 조회 또는 ⑤ 근거 수집 선택
builder.add_conditional_edges(
    "search_hybrid", after_hybrid, {"graph": "search_graph", "collect": "collect"},
)
builder.add_edge("search_graph", "collect")
builder.add_edge("collect", "answer")
builder.add_edge("answer", END)
routing_graph = builder.compile()


### 4. 실행과 근거 확인


In [ ]:
# 초기 상태에 새 질문을 넣어 그래프 실행
new_result = routing_graph.invoke({**initial_state, "question": new_question})


# 저장된 실행 결과에서 경로, 검색 입력, Cypher, 후보를 출력합니다.
print("선택 경로:", new_result["route"])
if new_result["route"] == "combined":
    print("줄거리 검색 입력:", new_result["semantic_query"])
    print("그래프 조회 입력:", new_result["graph_request"])
print("생성 Cypher:\n", new_result["cypher"])
display(pd.DataFrame(new_result["graph_rows"]))
display(pd.DataFrame([
    {"source_id": source_id, "title": documents_by_id[source_id]["title"]}
    for source_id in new_result["source_ids"]
]))
print("최종 답변:")
display(Markdown(new_result["answer"]))
print("인용 ID:", new_result["evidence_ids"])

# 인용 ID 대조와 원문 출력: 답변이 원문과 일치하는지는 직접 확인
evidence_by_id = {item["evidence_id"]: item for item in new_result["evidence"]}
inline_ids = set(re.findall(r"\[([^\[\]]+)\]", new_result["answer"]))
unknown_ids = (inline_ids | set(new_result["evidence_ids"])) - set(evidence_by_id)
print("제공하지 않은 인용 ID:", unknown_ids)
print("본문과 인용 목록의 차이:", inline_ids ^ set(new_result["evidence_ids"]))
for evidence_id in new_result["evidence_ids"]:
    if evidence_id in evidence_by_id:
        print("\n근거:", evidence_id)
        print(evidence_by_id[evidence_id]["text"])
